# Percolation Threshold Readability Experiment

This notebook demonstrates a novel machine learning method for text readability assessment.


In [ ]:
# Install dependencies
import subprocess, sys

def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2')

print('Dependencies installed')


In [ ]:
# Imports
import json
import re
import numpy as np
from collections import defaultdict, Counter
import sys
import gc
import matplotlib.pyplot as plt

def log(msg):
    print(f'[INFO] {msg}', flush=True)

print('Imports complete')


In [ ]:
# Data loading helper
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-703cc9-network-percolation-features-for-text-re/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print('Loaded data from GitHub URL')
            return data
    except Exception as e:
        print(f'GitHub URL load failed: {e}')
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            data = json.load(f)
            print('Loaded data from local file')
            return data
    raise FileNotFoundError('Could not load mini_demo_data.json')

print('Data loading helper defined')


In [ ]:
data = load_data()

texts = []
labels = []
for dataset in data.get('datasets', []):
    for example in dataset.get('examples', []):
        texts.append(example['input'])
        labels.append(int(example['output']))

print(f'Loaded {len(texts)} examples')
print(f'Grades: {sorted(set(labels))}')


In [ ]:
# Configuration
N_SAMPLES = min(len(texts), 48)
WINDOW_SIZE = 3
MIN_FREQ = 2
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2

print(f'Config: N_SAMPLES={N_SAMPLES}')


## Processing Steps

Now we'll implement the core methods:
1. **SimplePercolationNetwork** - Builds word co-occurrence networks
2. **SimpleBaselineReadability** - Computes traditional readability features
3. **SimpleLinearModel** - Linear regression from scratch
